# Simple Causal Narrative Analysis

This notebook demonstrates a minimal runnable example for analyzing causal narratives in political discourse using BERT models.

**Note**: This example uses BERT for Causal Detection and Span Extraction. Semantic Role Labeling (SRL) uses spaCy as AllenNLP (BERT-based SRL) might not be available in all environments.

In [1]:
import warnings
import sys
from loguru import logger

warnings.filterwarnings('ignore')
logger.remove()
logger.add(sys.stderr, level='INFO')

1

In [2]:

sentences = [
    "The Federal Reserve increased interest rates, resulting in prices deflating.",
    "The central bank raised the benchmark rate, resulting in the economy deflating severely.",
    "The bank increased the lending rate, resulting in the currency deflating significantly.",
    "The Federal Reserve increased interest rates, resulting in the financial system failing.",
    "The central bank raised the benchmark rate, resulting in the economy contracting severely.",
    "The bank increased the lending rate, resulting in the economy entering a recession.",
    "The government cut public spending, resulting in the economy receding.",
    "The administration reduced the budget, resulting in economic growth contracting.",
    "The state slashed funding, resulting in the economy slumping.",
    "The government cut public spending, resulting in infrastructure failing.",
    "The administration reduced the budget, resulting in public services failing.",
    "The state slashed funding, resulting in the healthcare system collapsing.",
    "The president delivered a speech about national unity yesterday.",
    "The general election is scheduled for next November."
]


In [3]:
from causal_narrative.detection import CausalDetector

logger.info("Initializing BERT Causal Detector...")
detector = CausalDetector(method='bert')

logger.info("Running detection...")
detection_results = detector.detect(sentences, show_progress=True)

causal_sentences = []
for i, res in enumerate(detection_results):
    if res.has_causality:
        print(f"Sentence {i+1} is CAUSAL (score: {res.score:.2f})")
        causal_sentences.append(sentences[i])
    else:
        print(f"Sentence {i+1} is NOT CAUSAL (score: {res.score:.2f})")

2026-03-25 17:21:10.372 | INFO     | __main__:<module>:3 - Initializing BERT Causal Detector...
2026-03-25 17:21:10.374 | INFO     | causal_narrative.detection:_resolve_bert_model_source:299 - Loading BERT model from local cache: model/causal-narrative_roberta-causal-narrative-classifier
2026-03-25 17:21:10.734 | INFO     | causal_narrative.detection:__init__:394 - BERTCausalDetector loaded from: model/causal-narrative_roberta-causal-narrative-classifier
2026-03-25 17:21:10.734 | INFO     | causal_narrative.detection:__init__:395 - Device: cpu
2026-03-25 17:21:10.734 | INFO     | causal_narrative.detection:__init__:678 - CausalDetector initialized with BERT method (model=causal-narrative/roberta-causal-narrative-classifier, cache_dir=model)
2026-03-25 17:21:10.735 | INFO     | __main__:<module>:6 - Running detection...
2026-03-25 17:21:10.735 | INFO     | causal_narrative.detection:_detect_bert_batch:883 - Starting batch BERT detection for 14 sentences
BERT detection: 100%|██████████| 

Sentence 1 is CAUSAL (score: 0.99)
Sentence 2 is CAUSAL (score: 0.99)
Sentence 3 is CAUSAL (score: 0.99)
Sentence 4 is CAUSAL (score: 0.99)
Sentence 5 is CAUSAL (score: 0.99)
Sentence 6 is CAUSAL (score: 0.99)
Sentence 7 is CAUSAL (score: 0.99)
Sentence 8 is CAUSAL (score: 0.99)
Sentence 9 is CAUSAL (score: 0.99)
Sentence 10 is CAUSAL (score: 0.99)
Sentence 11 is CAUSAL (score: 0.99)
Sentence 12 is CAUSAL (score: 0.99)
Sentence 13 is NOT CAUSAL (score: 0.01)
Sentence 14 is NOT CAUSAL (score: 0.01)


In [4]:
import pandas as pd
from causal_narrative.extraction import CausalSpanExtractor

logger.info("Initializing BERT Span Extractor...")
extractor = CausalSpanExtractor(method='bert')

logger.info("Extracting cause-effect spans...")
span_results = extractor.extract(causal_sentences, show_progress=True)

valid_spans = []
for i, span in enumerate(span_results):
    if span and span.cause_text and span.effect_text:
        valid_spans.append({
            'sentence': causal_sentences[i],
            'cause_text': span.cause_text,
            'effect_text': span.effect_text
        })
        print(f"\nSentence: {causal_sentences[i]}")
        print(f"  Cause:  {span.cause_text}")
        print(f"  Effect: {span.effect_text}")
    else:
        print(f"\nSentence: {causal_sentences[i]} - No span extracted")

df_spans = pd.DataFrame(valid_spans)
print(f"\nExtracted {len(df_spans)} valid causal spans.")

2026-03-25 17:21:15.403 | INFO     | __main__:<module>:4 - Initializing BERT Span Extractor...
2026-03-25 17:21:15.406 | INFO     | causal_narrative.extraction:_resolve_bert_model_source:544 - Loading BERT model from local cache: model/causal-narrative_roberta-causal-span-extractor
2026-03-25 17:21:15.556 | INFO     | causal_narrative.extraction:__init__:636 - BERTSpanExtractor loaded from: model/causal-narrative_roberta-causal-span-extractor
2026-03-25 17:21:15.556 | INFO     | causal_narrative.extraction:__init__:637 - Device: cpu
2026-03-25 17:21:15.557 | INFO     | causal_narrative.extraction:__init__:988 - CausalSpanExtractor initialized with BERT method (model=causal-narrative/roberta-causal-span-extractor, cache_dir=model)
2026-03-25 17:21:15.557 | INFO     | __main__:<module>:7 - Extracting cause-effect spans...
2026-03-25 17:21:15.557 | INFO     | causal_narrative.extraction:_extract_bert_batch:1218 - Starting batch BERT extraction for 12 sentences
BERT extraction: 100%|██████


Sentence: The Federal Reserve increased interest rates, resulting in prices deflating.
  Cause:  The Federal Reserve increased interest rates
  Effect: prices deflating

Sentence: The central bank raised the benchmark rate, resulting in the economy deflating severely.
  Cause:  The central bank raised the benchmark rate
  Effect: the economy deflating severely

Sentence: The bank increased the lending rate, resulting in the currency deflating significantly.
  Cause:  The bank increased the lending rate
  Effect: the currency deflating significantly

Sentence: The Federal Reserve increased interest rates, resulting in the financial system failing.
  Cause:  The Federal Reserve increased interest rates
  Effect: the financial system failing

Sentence: The central bank raised the benchmark rate, resulting in the economy contracting severely.
  Cause:  The central bank raised the benchmark rate
  Effect: the economy contracting severely

Sentence: The bank increased the lending rate, resu

In [5]:
from causal_narrative.semantic_role_labeling import get_srl

logger.info("Initializing SRL (spaCy)...")
srl = get_srl('spacy', model_name='en_core_web_sm')

logger.info("Running SRL on Cause and Effect spans...")
cause_srl_results = srl.process(df_spans['cause_text'].tolist())
effect_srl_results = srl.process(df_spans['effect_text'].tolist())

df_spans['cause_srl'] = cause_srl_results
df_spans['effect_srl'] = effect_srl_results

2026-03-25 17:21:20.793 | INFO     | __main__:<module>:3 - Initializing SRL (spaCy)...
2026-03-25 17:21:21.028 | INFO     | causal_narrative.semantic_role_labeling:__init__:487 - ✓ Loaded spaCy model: en_core_web_sm
2026-03-25 17:21:21.028 | INFO     | __main__:<module>:6 - Running SRL on Cause and Effect spans...
2026-03-25 17:21:21.030 | INFO     | causal_narrative.semantic_role_labeling:process:694 - Starting batch SRL processing for 12 texts using spaCy
2026-03-25 17:21:21.053 | INFO     | causal_narrative.semantic_role_labeling:process:745 - Completed batch SRL processing: 12/12 texts with verbs found
2026-03-25 17:21:21.054 | INFO     | causal_narrative.semantic_role_labeling:process:694 - Starting batch SRL processing for 12 texts using spaCy
2026-03-25 17:21:21.065 | INFO     | causal_narrative.semantic_role_labeling:process:745 - Completed batch SRL processing: 8/12 texts with verbs found


In [6]:
print("\n--- Sample SRL Results ---")
for i, row in df_spans.head(3).iterrows():
    print(f"\nSentence: {row['sentence']}")
    print(f"  Cause Span: {row['cause_text']}")
    print(f"  Cause SRL: {row['cause_srl']}")
    print(f"  Effect Span: {row['effect_text']}")
    print(f"  Effect SRL: {row['effect_srl']}")


--- Sample SRL Results ---

Sentence: The Federal Reserve increased interest rates, resulting in prices deflating.
  Cause Span: The Federal Reserve increased interest rates
  Cause SRL: {'words': ['The', 'Federal', 'Reserve', 'increased', 'interest', 'rates'], 'verbs': [{'verb': 'increase', 'description': '[ARG0: The Federal Reserve] [V: increase] [ARG1: interest rates]', 'tags': ['O', 'O', 'O', 'B-V', 'O', 'O']}]}
  Effect Span: prices deflating
  Effect SRL: {'words': ['prices', 'deflating'], 'verbs': []}

Sentence: The central bank raised the benchmark rate, resulting in the economy deflating severely.
  Cause Span: The central bank raised the benchmark rate
  Cause SRL: {'words': ['The', 'central', 'bank', 'raised', 'the', 'benchmark', 'rate'], 'verbs': [{'verb': 'raise', 'description': '[ARG0: The central bank] [V: raise] [ARG1: the benchmark rate]', 'tags': ['O', 'O', 'O', 'B-V', 'O', 'O', 'O']}]}
  Effect Span: the economy deflating severely
  Effect SRL: {'words': ['the', 'ec

In [7]:
from causal_narrative.embedding import (
    SentenceEmbedder,
    generate_role_based_embeddings,
    generate_phrase_embeddings
)
from causal_narrative.event_clustering import (
    run_hdbscan,
    generate_cluster_names_from_srl,
    generate_cluster_names_from_texts
)
from causal_narrative.semantic_role_labeling import is_event_srl
import numpy as np
import pandas as pd

# Initialize embedder
print("\nInitializing SentenceEmbedder...")
embedder = SentenceEmbedder()

# Check SRL validity
df_spans['cause_valid_for_role'] = df_spans['cause_srl'].apply(is_event_srl)
df_spans['effect_valid_for_role'] = df_spans['effect_srl'].apply(is_event_srl)

print(f"Cause valid for role: {df_spans['cause_valid_for_role'].sum()}/{len(df_spans)}")
print(f"Effect valid for role: {df_spans['effect_valid_for_role'].sum()}/{len(df_spans)}")

# Initialize cluster columns
df_spans['cause_cluster_id'] = -1
df_spans['cause_cluster_name'] = ''
df_spans['effect_cluster_id'] = -1
df_spans['effect_cluster_name'] = ''

# --- Cause Clustering ---
print("\n--- Cause Clustering ---")
# 1. Role-based
cause_role_mask = df_spans['cause_valid_for_role']
if cause_role_mask.any():
    print(f"Clustering {cause_role_mask.sum()} role-based causes...")
    cause_srl_list = df_spans.loc[cause_role_mask, 'cause_srl'].tolist()
    
    embeddings_cause_role = generate_role_based_embeddings(
        srl_results=cause_srl_list,
        embedder=embedder,
        batch_size=32,
        show_progress=False
    )
    
    # Use HDBSCAN
    c_ids, _ = run_hdbscan(embeddings_cause_role, min_cluster_size=2)
        
    names = generate_cluster_names_from_srl(
        labels=c_ids,
        srl_results=cause_srl_list,
        fallback_texts=df_spans.loc[cause_role_mask, 'cause_text'].tolist()
    )
    
    # No prefix
    
    df_spans.loc[cause_role_mask, 'cause_cluster_id'] = c_ids
    df_spans.loc[cause_role_mask, 'cause_cluster_name'] = [names[cid] for cid in c_ids]

# 2. Phrase-based
cause_phrase_mask = ~cause_role_mask
if cause_phrase_mask.any():
    print(f"Clustering {cause_phrase_mask.sum()} phrase-based causes...")
    cause_texts = df_spans.loc[cause_phrase_mask, 'cause_text'].tolist()
    
    embeddings_cause_phrase = generate_phrase_embeddings(
        texts=cause_texts,
        embedder=embedder,
        batch_size=32,
        show_progress=False
    )
    
    # Use HDBSCAN
    c_ids, _ = run_hdbscan(embeddings_cause_phrase, min_cluster_size=2)
        
    names = generate_cluster_names_from_texts(
        labels=c_ids,
        texts=cause_texts
    )
    
    # No prefix
    
    # Offset IDs
    max_id = df_spans['cause_cluster_id'].max()
    offset = max_id + 2
    c_ids_shifted = c_ids + offset
    
    df_spans.loc[cause_phrase_mask, 'cause_cluster_id'] = c_ids_shifted
    df_spans.loc[cause_phrase_mask, 'cause_cluster_name'] = [names[cid] for cid in c_ids]

# --- Effect Clustering ---
print("\n--- Effect Clustering ---")

# Determine global offset for effect clusters to avoid ID collision with causes
# We add a safe margin (e.g., 100)
max_cause_id = df_spans['cause_cluster_id'].max()
effect_id_offset = max_cause_id + 100
print(f"Applying offset {effect_id_offset} to effect clusters to avoid ID collision")

# 1. Role-based
effect_role_mask = df_spans['effect_valid_for_role']
if effect_role_mask.any():
    print(f"Clustering {effect_role_mask.sum()} role-based effects...")
    effect_srl_list = df_spans.loc[effect_role_mask, 'effect_srl'].tolist()
    
    embeddings_effect_role = generate_role_based_embeddings(
        srl_results=effect_srl_list,
        embedder=embedder,
        batch_size=32,
        show_progress=False
    )
    
    # Use HDBSCAN
    c_ids, _ = run_hdbscan(embeddings_effect_role, min_cluster_size=2)
        
    names = generate_cluster_names_from_srl(
        labels=c_ids,
        srl_results=effect_srl_list,
        fallback_texts=df_spans.loc[effect_role_mask, 'effect_text'].tolist()
    )
    
    # No prefix
    
    # Apply Offset
    c_ids_shifted = c_ids + effect_id_offset
    
    df_spans.loc[effect_role_mask, 'effect_cluster_id'] = c_ids_shifted
    df_spans.loc[effect_role_mask, 'effect_cluster_name'] = [names[cid] for cid in c_ids]

# 2. Phrase-based
effect_phrase_mask = ~effect_role_mask
if effect_phrase_mask.any():
    print(f"Clustering {effect_phrase_mask.sum()} phrase-based effects...")
    effect_texts = df_spans.loc[effect_phrase_mask, 'effect_text'].tolist()
    
    embeddings_effect_phrase = generate_phrase_embeddings(
        texts=effect_texts,
        embedder=embedder,
        batch_size=32,
        show_progress=False
    )
    
    # Use HDBSCAN
    c_ids, _ = run_hdbscan(embeddings_effect_phrase, min_cluster_size=2)
        
    names = generate_cluster_names_from_texts(
        labels=c_ids,
        texts=effect_texts
    )
    
    # No prefix
    
    # Offset IDs (based on current max effect ID)
    max_id = df_spans['effect_cluster_id'].max()
    offset = max_id + 2
    c_ids_shifted = c_ids + offset
    
    df_spans.loc[effect_phrase_mask, 'effect_cluster_id'] = c_ids_shifted
    df_spans.loc[effect_phrase_mask, 'effect_cluster_name'] = [names[cid] for cid in c_ids]

print("\n--- Clustering Results ---")
print("Cause Clusters:")
print(df_spans[['cause_text', 'cause_cluster_id', 'cause_cluster_name']].sort_values('cause_cluster_id').to_string())
print("\nEffect Clusters:")
print(df_spans[['effect_text', 'effect_cluster_id', 'effect_cluster_name']].sort_values('effect_cluster_id').to_string())


Initializing SentenceEmbedder...


2026-03-25 17:21:31.392 | INFO     | causal_narrative.embedding:__init__:91 - Loaded embedding model: sentence-transformers/all-MiniLM-L6-v2 (dimension: 384)
2026-03-25 17:21:31.395 | INFO     | causal_narrative.embedding:generate_role_based_embeddings:401 - Generating role-based embeddings for 12 SRL results


Cause valid for role: 12/12
Effect valid for role: 8/12

--- Cause Clustering ---
Clustering 12 role-based causes...


2026-03-25 17:21:32.862 | INFO     | causal_narrative.embedding:generate_role_based_embeddings:461 - Role-based embeddings generated: shape=(12, 1152)
2026-03-25 17:21:33.255 | INFO     | causal_narrative.event_clustering:run_hdbscan:441 - Running HDBSCAN: min_cluster_size=2, min_samples=2, metric=euclidean, n_samples=12


TypeError: check_array() got an unexpected keyword argument 'force_all_finite'

In [39]:
from causal_narrative.network import CausalNetworkBuilder
from causal_narrative.viz import visualize_causal_network

# Initialize builder
builder = CausalNetworkBuilder()

# Build network from DataFrame
builder.build_from_dataframe(
    df_spans,
    cause_col='cause_cluster_id',
    effect_col='effect_cluster_id',
    cause_text_col='cause_cluster_name',
    effect_text_col='effect_cluster_name'
)

# Set node labels explicitly
G = builder.graph
for node in G.nodes():
    # Try to find the name from cause clusters
    cause_match = df_spans[df_spans['cause_cluster_id'] == node]
    if not cause_match.empty:
        label = cause_match.iloc[0]['cause_cluster_name']
        G.nodes[node]['label'] = label
        continue
        
    # Try to find the name from effect clusters
    effect_match = df_spans[df_spans['effect_cluster_id'] == node]
    if not effect_match.empty:
        label = effect_match.iloc[0]['effect_cluster_name']
        G.nodes[node]['label'] = label
        continue
    
    # Fallback
    G.nodes[node]['label'] = f"Cluster {node}"

# Visualize network
visualize_causal_network(builder.graph, output_html='causal_network.html')
print("Causal network saved to causal_network.html")

2026-02-26 20:41:21.791 | INFO     | causal_narrative.network:build_from_dataframe:108 - Building network from DataFrame with 12 causal pairs...
2026-02-26 20:41:21.792 | INFO     | causal_narrative.network:build_from_dataframe:123 - Network built successfully: 6 nodes, 7 edges



Generating interactive network...
  Nodes: 6
  Edges: 7
  ✓ HTML saved: causal_network.html
Causal network saved to causal_network.html


## Causal network

